In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import argparse
import os
from unsloth import FastLanguageModel

os.environ["CUDA_VISIBLE_DEVICES"] = "5"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = "cuda" if torch.cuda.is_available() else "cpu"


/home/lab/biancaraimondi/LLM_Format/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


2025-03-11 16:15:52,217	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
def get_model(lora_path):
    max_seq_length = 2048 # Can increase for longer reasoning traces
    lora_rank = 32 # Larger rank = smarter, but slower
    # Load base model
    #lora_path = model_dir
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = lora_path,
        max_seq_length = max_seq_length,
        load_in_4bit = True, # False for LoRA 16bit
        fast_inference = True, # Enable vLLM fast inference
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.2, # Reduce if out of memory
    )
    model.eval()
    FastLanguageModel.for_inference(model)
    return model, tokenizer

def merge_and_save_lora(base_model_path, lora_model_path, output_dir):
    model, tokenizer = get_model(lora_model_path)
    model.save_pretrained_merged(output_dir, tokenizer)

In [4]:
import os
models_B = ["0.5", "1.5", "3"]
checkpoints = ["500", "1000", "1500"]
one_shots = ["", "_one_shot"]

for model_B in models_B:
    for checkpoint in checkpoints:
        for one_shot in one_shots:
            pretrained_model = "Qwen/Qwen2.5-Coder-" + model_B + "B-Instruct"
            model_dir = "Qwen-" + model_B + "B" + one_shot + "/checkpoint-" + checkpoint
            merged_model_dir = "merged_models/" + model_dir

            if not os.path.exists(merged_model_dir):
                print(f"Generating merged model for {model_dir}...")
                # model, tokenizer = get_models(pretrained_model, model_dir)
                # merge_and_save_lora(tokenizer, model, merged_model_dir)
                merge_and_save_lora(base_model_path=pretrained_model, lora_model_path=model_dir, output_dir=merged_model_dir)

Generating merged model for Qwen-0.5B/checkpoint-500...
INFO 03-11 16:17:06 __init__.py:207] Automatically detected platform cuda.
==((====))==  Unsloth 2025.2.15: Fast Qwen2 patching. Transformers: 4.49.0.
   \\   /|    GPU: NVIDIA A100-SXM4-80GB. Max memory: 79.151 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-coder-0.5b-instruct-bnb-4bit with actual GPU utilization = 19.9%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 79.15 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 256.
Unsloth: vLLM's KV Cache can use up to 15.25 GB. Also swap space = 6 GB.
INFO 03-11 16:17:20 config.py:549] This model supports multiple

[W311 16:17:22.145745822 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())


INFO 03-11 16:17:22 loader.py:1089] Loading weights with BitsAndBytes quantization.  May take a while ...
INFO 03-11 16:17:23 weight_utils.py:254] Using model weights format ['*.safetensors']
INFO 03-11 16:17:24 weight_utils.py:270] Time spent downloading weights for unsloth/qwen2.5-coder-0.5b-instruct-bnb-4bit: 1.060545 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.27s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:02<00:00,  2.27s/it]

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  5.95it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  5.90it/s]



INFO 03-11 16:17:27 model_runner.py:1115] Loading model weights took 0.4342 GB
INFO 03-11 16:17:27 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 03-11 16:17:35 worker.py:267] Memory profiling takes 7.40 seconds
INFO 03-11 16:17:35 worker.py:267] the current vLLM instance can use total_gpu_memory (79.15GiB) x gpu_memory_utilization (0.20) = 15.75GiB
INFO 03-11 16:17:35 worker.py:267] model weights take 0.43GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.41GiB; the rest of the memory reserved for KV Cache is 13.82GiB.
INFO 03-11 16:17:35 executor_base.py:111] # cuda blocks: 75455, # CPU blocks: 32768
INFO 03-11 16:17:35 executor_base.py:116] Maximum concurrency for 2048 tokens per request: 589.49x
INFO 03-11 16:17:43 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory erro

...

Capturing CUDA graph shapes: 100%|██████████| 31/31 [00:50<00:00,  1.64s/it]

INFO 03-11 16:51:23 model_runner.py:1562] Graph capturing finished in 51 secs, took 0.09 GiB
INFO 03-11 16:51:23 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 55.86 seconds


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 720.01 out of 1007.38 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 36/36 [00:01<00:00, 19.25it/s]


Unsloth: Saving tokenizer... Done.
Done.
